# Alluvial Diagrams: Showing How Categories Flow Together

**Optional, for the Project 2 work session (Mon, Nov. 23).** This is a worked example, not a lab: nothing to submit and nothing graded. It's worth a look if your project has several categorical variables and your story is about how groups split or combine, or if you want one more interactive chart type for your notebook.

## Objectives

By the end of this lab, you should be able to:

- Explain what an alluvial diagram shows, and how it differs from the tree charts in Lab 9
- Build one with `px.parallel_categories`, and choose the order of the axes and the categories so it reads clearly
- Use color to tell one story, instead of coloring everything
- Use the diagram's built-in interactions (dragging, hovering) to explore
- Decide when an alluvial diagram is the right choice and when a heatmap or bar chart works better

## Setup

We'll use two datasets: the Titanic passenger list (891 passengers: class, sex, age group, and whether they survived), and the superstore sales data from Lab 9.

In [ ]:
# !pip install pandas plotly

import pandas as pd
import plotly.express as px

titanic = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")
sales = pd.read_csv("superstore.csv")

titanic[["class", "sex", "who", "embark_town", "alive", "survived"]].head()

## 1. What an alluvial diagram shows

An **alluvial diagram** (also called a **parallel sets** diagram) has one vertical axis for each **categorical** variable. Each axis is split into blocks, one per category, sized by how many rows fall in that category. **Bands** connect the blocks: a band is a group of rows that share the same combination of categories, and its width is how many rows are in that group.

Read it left to right: follow a band and you can see how one group *splits* across the next variable, and how groups from different starting points *merge*.

Compare this to the tree charts from Lab 9. A treemap needs a strict hierarchy: every sub-category sits inside exactly one category. Alluvial diagrams don't. Every passenger has a class **and** a sex **and** an outcome, and none of those nests inside another, so the bands are free to cross.

You may also see **Sankey diagrams**, a close relative. They show flows between nodes (energy moving through a power grid, money moving between accounts), where the same node can receive flows from many places. In Plotly, the alluvial diagram is `px.parallel_categories`.

## 2. A first alluvial diagram

`px.parallel_categories` needs a data frame and a list of `dimensions`: the categorical columns to use as axes, in order from left to right.

In [ ]:
# make an alluvial diagram of the titanic data with the axes class -> sex -> alive
fig = px.parallel_categories(titanic, dimensions=["class", "sex", "alive"])
fig.show()

<div class="alert alert-warning">
<b>Stop and think:</b>

Hover over a few bands, then answer:

1. What is the hardest thing to read on this chart?
2. Can you tell, at a glance, whether women in first class were more likely to survive than men in third class? What would make that easier?
</div>

## 3. Color tells the story

Right now every band is the same color, so your eye has nothing to follow. Color one variable, and the bands keep that color as they split and merge across every axis.

Which variable to color depends on your question. If the question is *"who survived?"*, color by survival: then each block on the `class` and `sex` axes shows its mix of survivors and non-survivors.

`color` must be numeric, so we use the 0/1 `survived` column. Following the explanatory techniques from class, we'll make the outcome we care about stand out (survivors in blue) and push the rest into the background (grey).

In [ ]:
fig = px.parallel_categories(
    titanic,
    dimensions=["class", "sex", "alive"],
    color="survived",
    # survivors (1) blue, everyone else (0) grey
    color_continuous_scale=[[0, "#bbbbbb"], [1, "#0072B2"]],
    labels={"class": "Ticket class", "sex": "Sex", "alive": "Survived?"},
    title="Most women in first and second class survived; most men in every class did not",
)
fig.update_layout(coloraxis_showscale=False)  # the color bar adds nothing here
fig.show()

<div class="alert alert-info">
<b>Talk to a neighbor:</b>

1. Why did we hide the color bar?
2. Why is the outcome (`alive`) the **last** axis, not the first?
</div>

## 4. Order matters: axes and categories

Two orderings change how easy an alluvial diagram is to read:

- **Axis order.** Put the variable you're explaining (the outcome) at one end, and the variables that explain it before it. Put variables that interact strongly next to each other, so their bands don't cross more than they need to.
- **Category order within an axis.** Plotly sorts categories by how often they appear, which can scramble a natural order like First → Second → Third.

You can set the category order after making the figure with `fig.update_traces(dimensions=[...])`, passing one dictionary per axis, in the same left-to-right order. An empty dictionary `{}` leaves that axis alone.

Plotly also uses straight bands by default; `line_shape="hspline"` curves them, which makes crossings easier to follow.

In [ ]:
fig = px.parallel_categories(
    titanic,
    dimensions=["class", "sex", "alive"],
    color="survived",
    color_continuous_scale=[[0, "#bbbbbb"], [1, "#0072B2"]],
    labels={"class": "Ticket class", "sex": "Sex", "alive": "Survived?"},
    title="Most women in first and second class survived; most men in every class did not",
)
fig.update_layout(coloraxis_showscale=False)

# order class as First, Second, Third, and alive as yes, no
fig.update_traces(
    dimensions=[
        dict(categoryorder="array", categoryarray=["First", "Second", "Third"]),
        {},
        dict(categoryorder="array", categoryarray=["yes", "no"]),
    ],
    line_shape="hspline",
)
fig.show()

## 5. Built-in interactions

Alluvial diagrams come with interactions you don't have to code. On the chart above, try:

- **Hover** over a band to see how many passengers are in it, or over a block to see its total.
- **Drag an axis** sideways to change the axis order. Try moving `sex` to the left of `class`.
- **Drag a category block** up or down within an axis to reorder it.

Notice how much the picture changes when you reorder. The interaction is useful for *exploring*, to find the order that tells the clearest story, but for an explanatory chart you should **choose the order yourself in code** so every reader sees the same thing.

<div class="alert alert-warning">
<b>Stop and think:</b>

Drag `sex` to the far left. Is the survival story easier or harder to read than with `class` first? Why?
</div>

## 6. When an alluvial diagram is the wrong choice

Alluvial diagrams are not always the best chart for categorical data. Let's look at a case where they struggle: sales orders in the superstore data by **region** and **customer segment**.

These two variables cross (every region has every segment), so they suit an alluvial diagram in principle. Let's see how it reads.

In [ ]:
fig = px.parallel_categories(
    sales,
    dimensions=["region", "segment"],
    labels={"region": "Region", "segment": "Customer segment"},
    title="Orders by region and customer segment",
)
fig.update_traces(line_shape="hspline")
fig.show()

Now the same counts as a heatmap. Each cell is one region–segment combination, colored by the number of orders.

In [ ]:
# count orders for each region and segment combination, as a table with
# regions as rows and segments as columns
counts = pd.crosstab(sales["region"], sales["segment"])

fig = px.imshow(
    counts,
    text_auto=True,
    color_continuous_scale="Blues",
    labels={"x": "Customer segment", "y": "Region", "color": "Orders"},
    title="Orders by region and customer segment",
)
fig.show()

<div class="alert alert-info">
<b>Talk to a neighbor:</b>

Which chart makes it easier to compare the number of *Corporate* orders in the West and Ontario? Which makes it easier to see the overall shape of the data?
</div>

**Rules of thumb:**

- Use an **alluvial diagram** when a few combinations dominate (the data are *uneven*), when you have **three or more** categorical variables, or when the story is about groups **splitting and recombining** (like the Titanic).
- Use a **heatmap** (or grouped bar chart) when counts are spread fairly evenly across combinations, when you only have **two** variables, or when the reader needs to compare exact values.
- Keep each axis to a **handful of categories**. With many categories the bands turn into spaghetti; keep the largest few and group the rest into "Other".

## Try it on your data

Choose **one**:

- **Option A (recommended):** make an alluvial diagram from **your Project 2 dataset**, using 2–4 categorical variables (bin a numeric variable into a few categories with `pd.cut` if you need to). Will it make it into your notebook or final piece?
- **Option B:** make an alluvial diagram of the Titanic data with the axes `who` (man, woman, child) → `class` → `alive`. Drop rows with missing values first if you add `embark_town`.

Either way, choose your axis order, category order, and coloring on purpose. Then ask yourself: what does the diagram show, and would a heatmap or bar chart show it better? If you use one in your notebook, add the 1–2 sentence note saying what the reader can do with it.

In [ ]:
# Option B example
fig = px.parallel_categories(
    titanic,
    dimensions=["who", "class", "alive"],
    color="survived",
    color_continuous_scale=[[0, "#bbbbbb"], [1, "#0072B2"]],
    labels={"who": "Passenger", "class": "Ticket class", "alive": "Survived?"},
    title="Women and children survived at higher rates than men in every class",
)
fig.update_layout(coloraxis_showscale=False)
fig.update_traces(
    dimensions=[
        dict(categoryorder="array", categoryarray=["child", "woman", "man"]),
        dict(categoryorder="array", categoryarray=["First", "Second", "Third"]),
        dict(categoryorder="array", categoryarray=["yes", "no"]),
    ],
    line_shape="hspline",
)
fig.show()

## Acknowledgments

Adapted from R.C. Alvarado's *Alluvial Diagrams with Plotly* notebook (DS 2023, UVA).